In [2]:
import sys
import os
import math
import numpy as np
states   = {"s": 0, "E": 1, "5": 2, "I": 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([
    [0.0,  1.0,  0.0,  0.0,  0.0],
    [0.0,  0.9,  0.1,  0.0,  0.0],
    [0.0,  0.0,  0.0,  1.0,  0.0],
    [0.0,  0.0,  0.0,  0.9,  0.1],
    [0.0,  0.0,  0.0,  0.0,  0.0],
])
emission_nuc_codes = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

emission_probs = np.array([
    [0.00,  0.00,  0.00,  0.00],   # s  (silent)
    [0.25,  0.25,  0.25,  0.25],   # E  (uniform)
    [0.05,  0.00,  0.95,  0.00],   # 5  (mostly G)
    [0.40,  0.10,  0.10,  0.40],   # I  (AT-rich)
    [0.00,  0.00,  0.00,  0.00],   # e  (silent)
])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

NEG_INF = float('-inf')

def safe_log(x):
    """Return log(x), or -inf if x <= 0."""
    return math.log(x) if x > 0 else NEG_INF

def get_log_prob_for_state_path(state_path, sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        trans = state_transition_prob[states[state_path[i-1]]][states[state_path[i]]]
        emis  = emission_probs[states[state_path[i]]][emission_nuc_codes[sequence[i]]]
        res  += math.log(trans * emis)
    return res

k1     = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", query_sequence) + math.log(0.1)
k2     = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", query_sequence) + math.log(0.1)
k3     = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", query_sequence) + math.log(0.1)
k4     = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", query_sequence) + math.log(0.1)
k5     = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", query_sequence) + math.log(0.1)
k6     = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", query_sequence) + math.log(0.1)
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", query_sequence) + math.log(0.1)

print("Reference log-probabilities (manual paths):")
for name, val in [("k1  (5'ss @ pos  6)", k1), ("k2  (5'ss @ pos  8)", k2),
                  ("k3  (5'ss @ pos 12)", k3), ("k4  (5'ss @ pos 15)", k4),
                  ("k5  (5'ss @ pos 18)", k5), ("k6  (5'ss @ pos 22)", k6),
                  ("only_E             ", only_E)]:
    print(f"  {name}: {val:.6f}")

# Matrix Initialization
n_states = len(states)
n_obs    = len(query_sequence)

viterbi_value_matrix = np.full((n_states, n_obs), NEG_INF)
viterbi_trace_matrix = np.zeros((n_states, n_obs), dtype=int)

for j in range(n_states):
    t_prob = state_transition_prob[0][j]
    e_prob = emission_probs[j][emission_nuc_codes[query_sequence[0]]]
    viterbi_value_matrix[j][0] = safe_log(t_prob) + safe_log(e_prob)
    viterbi_trace_matrix[j][0] = 0

# Recursion

def calculate_prob_for_a_node(j, t):
    """
    Return (max_log_prob, best_prev_state_index) for state j at position t.

    V[j][t] = max_i( V[i][t-1] + log P(i->j) ) + log P(emit seq[t] | j)
    """
    e_prob = emission_probs[j][emission_nuc_codes[query_sequence[t]]]
    if e_prob <= 0:
        return NEG_INF, 0

    log_emis  = math.log(e_prob)
    max_val   = NEG_INF
    best_prev = 0

    for i in range(n_states):
        t_prob = state_transition_prob[i][j]
        prev   = viterbi_value_matrix[i][t - 1]
        if t_prob <= 0 or prev == NEG_INF:
            continue
        val = prev + math.log(t_prob) + log_emis
        if val > max_val:
            max_val, best_prev = val, i

    return max_val, best_prev


for t in range(1, n_obs):
    for j in range(n_states):
        viterbi_value_matrix[j][t], viterbi_trace_matrix[j][t] = calculate_prob_for_a_node(j, t)

# Termination: add final transition to end state 'e'

END_STATE = states["e"]
final_scores = np.full(n_states, NEG_INF)
for j in range(n_states):
    t_to_e = state_transition_prob[j][END_STATE]
    v      = viterbi_value_matrix[j][n_obs - 1]
    if t_to_e > 0 and v != NEG_INF:
        final_scores[j] = v + math.log(t_to_e)

# Traceback
def traceback_viterbi():
    """
    Recover the most-probable state path.
    Starts from argmax of final_scores (which includes the end-state transition),
    then walks backwards through viterbi_trace_matrix.
    """
    best_last_state = int(np.argmax(final_scores))
    best_log_prob   = final_scores[best_last_state]

    path = [best_last_state]
    for t in range(n_obs - 1, 0, -1):
        path.append(viterbi_trace_matrix[path[-1]][t])
    path.reverse()

    state_path = ''.join(id2state[s] for s in path)
    return state_path, best_log_prob


best_path, best_log_prob = traceback_viterbi()

# Print Results

state_labels = ["s", "E", "5", "I", "e"]
header = "      " + "".join(f"{c:>8}" for c in query_sequence)

print("\n" + "═" * 65)
print("VITERBI VALUE MATRIX  (log scale)")
print("═" * 65)
print(header)
for i, label in enumerate(state_labels):
    row = f"{label:<6}" + "".join(
        f"{viterbi_value_matrix[i][t]:>8.3f}" if viterbi_value_matrix[i][t] != NEG_INF
        else f"{'  -inf':>8}"
        for t in range(n_obs)
    )
    print(row)

print("\n" + "═" * 65)
print("FINAL SCORES  (last column + log( trans[j → e] ))")
print("═" * 65)
for j in range(n_states):
    v = final_scores[j]
    print(f"  state {id2state[j]}: {v:.6f}" if v != NEG_INF else f"  state {id2state[j]}: -inf")

print("\n" + "═" * 65)
print("VITERBI TRACE MATRIX  (best previous-state index)")
print("═" * 65)
print(header)
for i, label in enumerate(state_labels):
    row = f"{label:<6}" + "".join(f"{viterbi_trace_matrix[i][t]:>8}" for t in range(n_obs))
    print(row)

print("\n" + "═" * 65)
print("VITERBI RESULT")
print("═" * 65)
print(f"Sequence  : {query_sequence}")
print(f"Best path : {best_path}")
print(f"Log-prob  : {best_log_prob:.6f}")
print()

if '5' in best_path:
    ss_pos = best_path.index('5')
    print(f"5' splice site predicted at position {ss_pos}  "
          f"(nucleotide '{query_sequence[ss_pos]}')")
    print(f"  Exon   region : {query_sequence[:ss_pos]}")
    print(f"  Intron region : {query_sequence[ss_pos + 1:]}")
else:
    print("No splice site — full sequence is Exon.")

print()
print("Comparison with manually computed paths:")
for name, val in [("k1  EEEEEE5III…          ", k1),
                  ("k2  EEEEEEEE5II…          ", k2),
                  ("k3  EEEEEEEEEEEE5I…       ", k3),
                  ("k4  EEEEEEEEEEEEEEE5I…    ", k4),
                  ("k5  EEEEEEEEEEEEEEEEEE5I… ", k5),
                  ("k6  EEEEEEEEEEEEEEEEEEEEEE5…", k6),
                  ("only_E                   ", only_E)]:
    match = " ← MATCHES VITERBI" if abs(val - best_log_prob) < 0.001 else ""
    print(f"  {name}: {val:.6f}{match}")
print(f"  Viterbi best                           : {best_log_prob:.6f}  ← highest valid path")

Reference log-probabilities (manual paths):
  k1  (5'ss @ pos  6): -43.897400
  k2  (5'ss @ pos  8): -43.451113
  k3  (5'ss @ pos 12): -43.944833
  k4  (5'ss @ pos 15): -42.582256
  k5  (5'ss @ pos 18): -41.219678
  k6  (5'ss @ pos 22): -41.713398
  only_E             : -40.980251

═════════════════════════════════════════════════════════════════
VITERBI VALUE MATRIX  (log scale)
═════════════════════════════════════════════════════════════════
             C       T       T       C       A       T       G       T       G       A       A       A       G       C       A       G       A       C       G       T       A       A       G       T       C       A
s         -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf    -inf
E       -1.386  -2.878  -4.370  -5.861  -7.353  -8.845 -10.336 -11.828 -13.320 -14.811 -16.303 -17.794 -19.286 -20.778 -2